# Projeto Análise de Dados - Festival de Música

## PASSO 1: Inspeção Inicial

In [ ]:
import pandas as pd 
import numpy as np

# cria variavel para armazenar a planilha
df_festival = pd.read_csv('vendas_festival_bruto.csv')

# visualiza as primeiras linhas da planilha
df_festival.head()

In [15]:
# verifica o número total de linhas e colunas
df_festival.shape
df_festival.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_compra         120 non-null    str    
 1   id_cliente        120 non-null    str    
 2   idade_cliente     116 non-null    float64
 3   genero            119 non-null    str    
 4   estado_origem     120 non-null    str    
 5   setor_ingresso    120 non-null    str    
 6   lote              120 non-null    str    
 7   tipo_categoria    120 non-null    str    
 8   valor_pago        120 non-null    float64
 9   meio_pagamento    120 non-null    str    
 10  data_hora_compra  120 non-null    str    
dtypes: float64(2), str(9)
memory usage: 10.4 KB


## PASSO 2: Diagnóstico e Qualidade dos Dados

In [16]:
# verifica se possui linhas duplicadas na planilha e soma a quantidade (compara a linha inteira)
df_festival.duplicated().sum()

# verifica se possui duplicidade em coluna específica de id_compra (não pode haver mais de uma compra com o mesmo ID)
df_festival.duplicated(subset=['id_compra']).sum()

np.int64(0)

In [ ]:
# verifica as principais métricas através do comando .describe()
df_festival['idade_cliente'].describe()

count    116.000000
mean      51.982759
std      155.125530
min       17.000000
25%       22.000000
50%       25.500000
75%       31.000000
max      999.000000
Name: idade_cliente, dtype: float64

INCONSISTÊNCIA DE TEXTOS
Colunas de categoria como 'Meio de pagamento', 'Setor' ou 'Gênero' podem vir da base de dados com sujeira:
- Variações como "PIX", "pix", "Pix"
- Espaços extras no inicio ou no fim " Cartão", "Cartão "

Se não tratar isso, o Pandas irá contar "PIX" e "pix" como meios de pagamento diferentes, causando erro na visualização de dados final.

In [29]:
# conta quantas vezes aparece cada tipo de pagamento
df_festival['meio_pagamento'].value_counts()

meio_pagamento
PIX                              63
Cartão de Crédito - Parcelado    34
Cartão de Crédito - Vista        19
Boleto                            4
Name: count, dtype: int64

## PASSO 3: Limpeza e Tratamento dos Dados
Etapa para corrigir os problemas que foram identificados nos passos anteriores.
Ponto de atenção: não deve-se sobrescrever a base de dados original de forma a deletar os dados sem antes criar uma cópia ou planejar a ação.

1) Tratando a coluna 'idade_cliente' o Outlier (valor atípico):
Através do comando .describe(), vi que existe o valor '999' como idade de um cliente preenchida, o que estaria fora da regra de negócio. Porém, o método describe() retorna apenas os valores extremos de mínimo e máximo, então preciso garantir que além do máximo '999' não exista outro dado nessa coluna com erro de digitação.

In [ ]:
# verifica se possui mais idades acima de 100
df_festival[df_festival['idade_cliente'] > 100][['idade_cliente']]

,idade_cliente
7,999.0
31,999.0
65,999.0


In [33]:
# cria uma tabela copia para manter rastreabilidade dos dados originais e tratar os dados na copia
df_festival_limpo = df_festival.copy()

# substitui o outlier por um valor nulo para tratar com os demais nulos depois
df_festival_limpo['idade_cliente'] = df_festival_limpo['idade_cliente'].replace(999, np.nan)

In [37]:
# roda o describe() de novo para validar se a idade maxima 999 mudou
df_festival_limpo.describe()

,idade_cliente,valor_pago
count,113.000000,120.000000
mean,26.840708,701.775000
std,6.941807,503.221487
min,17.000000,180.000000
25%,22.000000,234.000000
50%,25.000000,468.000000
75%,30.000000,1250.000000
max,50.000000,1625.000000


Para tratar os nulos na coluna 'idade_cliente' será preenchido com o valor da media das idades. 
O motivo: Se escolhesse preencher os nulos com 'Não informado', estaria misturando uma coluna numérica com texto, e isso faria com que o Pandas convertesse a coluna inteira para texto (object). Isso não deixaria depois fazer contas matemáticas com a idade. 

In [39]:
# preenchendo os campos nulos da coluna 'idade_cliente' com a media das idades
media_idades = df_festival_limpo['idade_cliente'].mean()
df_festival_limpo['idade_cliente'] = df_festival_limpo['idade_cliente'].fillna(media_idades)

In [42]:
# verifica se a coluna 'idade_cliente' ainda possui nulos
df_festival_limpo.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_compra         120 non-null    str    
 1   id_cliente        120 non-null    str    
 2   idade_cliente     120 non-null    float64
 3   genero            119 non-null    str    
 4   estado_origem     120 non-null    str    
 5   setor_ingresso    120 non-null    str    
 6   lote              120 non-null    str    
 7   tipo_categoria    120 non-null    str    
 8   valor_pago        120 non-null    float64
 9   meio_pagamento    120 non-null    str    
 10  data_hora_compra  120 non-null    str    
dtypes: float64(2), str(9)
memory usage: 10.4 KB


2) Tratando a coluna 'genero' o valor nulo: Depois de ter tratado o outlier e os nulos da coluna 'idade_cliente', vi que a coluna 'genero' ainda possui um valor nulo que precisa ser tratado. Aqui, a escolha foi substituir o dado ausente por 'Não informado' em vez de deletar a linha, para evitar perda de registros.

In [43]:
# preenchendo os campos nulos da coluna 'genero' com 'Não informado'
df_festival_limpo['genero'] = df_festival_limpo['genero'].fillna('Não informado')

# PASSO 4: Análise Exploratória de Dados (EDA)
## Após limpar e tratar os dados nulos no passo anterior, inicia a etapa de extrair valor de negócio e responder às perguntas do festival. Irei começar explorando o comportamento de compras dos clientes.

In [69]:
# contagem para saber qual setor vendeu mais ingresso
df_festival_limpo.groupby('setor_ingresso').count().sort_values(by='id_compra',ascending=False)

,id_compra,id_cliente,idade_cliente,genero,estado_origem,lote,tipo_categoria,valor_pago,meio_pagamento,data_hora_compra
setor_ingresso,,,,,,,,,,
Pista Comum,48,48,48,48,48,48,48,48,48,48
Lounge VIP,25,25,25,25,25,25,25,25,25,25
Pista Premium,25,25,25,25,25,25,25,25,25,25
Passaporte 3 Dias,22,22,22,22,22,22,22,22,22,22


In [ ]:
# visualizar rapidamente as principais estatísticas sobre a tabela
df_festival_limpo.describe()

,idade_cliente,valor_pago
count,120.000000,120.000000
mean,26.840708,701.775000
std,6.734542,503.221487
min,17.000000,180.000000
25%,22.000000,234.000000
50%,26.000000,468.000000
75%,30.000000,1250.000000
max,50.000000,1625.000000


In [71]:
# agrupa pelo setor e calcula o valor medio pago em cada um
df_festival_limpo.groupby('setor_ingresso')['valor_pago'].mean()

setor_ingresso
Lounge VIP           1490.000000
Passaporte 3 Dias     897.590909
Pista Comum           267.375000
Pista Premium         575.280000
Name: valor_pago, dtype: float64

# PASSO 5: Exportação e salvamento do Projeto
## Nesta etapa a base limpa e tratada será salva para alimentar dashboards ou relatórios finais.

In [72]:
# salva a tabela de dados limpa e tratada como um novo arquivo CSV
df_festival_limpo.to_csv('vendas_festival_limpo.csv',index=False)

# PASSO 6: Validação pós-tratamento
## Após os passos acima, gerei um novo arquivo csv com a base limpa, mas ao visualizar a nova planilha ainda identifiquei que haviam dados inconsistentes. Por esse motivo, vi a necessidade de realizar uma nova validação dos dados.

1) Para regra de negócio, irei garantir que não exista o preenchimento de idades de clientes abaixo de 0 e maior do que 100. Primeiro identifico as linhas que estão dentro dessa faixa, depois substituo por valor nulo e por fim calculo a media para preencher o campo.

In [ ]:
# localiza as linhas da coluna com idades menores iguais a 0 ou maiores que 100 e preenche com nulo
df_festival_limpo.loc[
                        (df_festival_limpo['idade_cliente'] <= 0) | 
                        (df_festival_limpo['idade_cliente'] > 100),
                        'idade_cliente'] = np.nan 

In [84]:
# calcula a media das idades e preenche os nulos com a media
media_idade = df_festival_limpo['idade_cliente'].mean()
df_festival_limpo['idade_cliente'] = df_festival_limpo['idade_cliente'].fillna(media_idade)

In [87]:
# arredondamento para 1 casa decimal na idade
df_festival_limpo['idade_cliente'] = df_festival_limpo['idade_cliente'].round(0)

# exibe dados para validar após tratar os dados da coluna de idade
df_festival_limpo[['idade_cliente']]

,idade_cliente
0,24.0
1,31.0
2,27.0
3,24.0
4,19.0
...,...
115,24.0
116,28.0
117,22.0
118,36.0


In [89]:
# exporta novamente a tabela e trata o separador por ponto e virgula e usa virgula como decimal
df_festival_limpo.to_csv('vendas_festival.csv', index=False, sep=';', decimal=',')